# Le Gros Chaton — Qwen2.5-Coder-7B QLoRA Fine-Tune

Fine-tune Qwen2.5-Coder-7B for coding agent tasks using our RLVR pipeline.
Runs on **Kaggle L4 24GB** with 4-bit QLoRA.

**Target:** Rival GLM-5.2 / Opus 4.7 on Terminal-Bench, SWE-Bench
**Method:** QLoRA + self-play data + GRPO with proportional rewards

In [ ]:
# 1. Check GPU
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 2. Clone repo + install deps
import os
REPO = 'https://github.com/Mateooo93/le-gros-chaton.git'
if not os.path.exists('le-gros-chaton'):
    !git clone {REPO}
%cd le-gros-chaton
!pip install -q transformers accelerate peft bitsandbytes trl
!pip install -q tiktoken tokenizers datasets

In [ ]:
# 3. Set HF token (get from huggingface.co/settings/tokens)
import os
HF_TOKEN = ""  # Paste your HF write token here
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['CHATON_HF_REPO'] = 'mateo0093/le-fat-chaton-ckpt'

In [ ]:
# 4. Load Qwen2.5-Coder-7B with QLoRA (4-bit)
import sys
sys.path.insert(0, '.')

from finetune_qwen import load_model

model, tokenizer = load_model(
    'Qwen/Qwen2.5-Coder-7B',
    use_lora=True,
    use_4bit=True,
    device='auto',
)
print('Model loaded successfully!')

In [ ]:
# 5. Load HumanEval problems
from eval.humaneval_loader import load as load_humaneval
problems_raw = load_humaneval(limit=10)
problems = [{
    'id': p.id, 'prompt': p.prompt, 'tests': p.tests,
    'entry_point': p.entry_point,
} for p in problems_raw]
print(f'Loaded {len(problems)} problems')
print(f'Sample prompt: {problems[0]["prompt"][:100]}...')

In [ ]:
# 6. Test generation
prompt = problems[0]['prompt']
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
out = model.generate(
    **inputs, max_new_tokens=128,
    temperature=0.8, top_p=0.95, do_sample=True,
)
solution = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f'Generated: {solution[:200]}')

In [ ]:
# 7. Generate self-play training data
from finetune_qwen import collect_self_play_data

data = collect_self_play_data(
    model, tokenizer, problems,
    out_path='self_play_qwen.json',
    n_attempts=3, max_tokens=512,
)
print(f'Generated {len(data)} training examples')

In [ ]:
# 8. RLVR Training
from finetune_qwen import train_rlvr

train_rlvr(
    model, tokenizer, problems,
    n_steps=200, lr=2e-5, batch_size=4,
    out_path='qwen_rlvr_final',
)

In [ ]:
# 9. Test the fine-tuned model
model.eval()
prompt = problems[0]['prompt']
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
out = model.generate(
    **inputs, max_new_tokens=256,
    temperature=0.6, top_p=0.95, do_sample=False,
)
solution = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f'Before fine-tune: solution empty?')
print(f'\nFinal solution:\n{solution}')

In [ ]:
# 10. Evaluate on HumanEval
from eval.eval import evaluate
from model import GPT  # our eval harness uses GPT.generate - need to adapt
print('Run: python eval/eval.py --ckpt qwen_rlvr_final --n 20 --ks 1 5 --limit 10')